In [2]:
import pandas as pd

df = pd.read_csv("D:\\Business Analyst\\Python\\ml_assesment\\Data\\q3_retail_promotions.csv")#loading data
df['transaction_date'] = pd.to_datetime(df['transaction_date']) # date time conversion
# extracting year, month and day_of_week
df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek

# Month-end flag
df['is_month_end'] = df['transaction_date'].dt.day.apply(lambda x: 1 if x >= 25 else 0)

print(df.head())

  transaction_date  store_id store_size location_type  promotion_type  \
0       2022-01-01        28      small    semi-urban       free_gift   
1       2022-01-01         5     medium    semi-urban       free_gift   
2       2022-01-02        13      small    semi-urban  loyalty_points   
3       2022-01-02        17      small         urban       free_gift   
4       2022-01-03        50     medium    semi-urban            bogo   

   is_weekend  is_festival  competition_density  items_sold  year  month  \
0           1            0                    5         224  2022      1   
1           1            1                    1         348  2022      1   
2           1            0                    6         249  2022      1   
3           1            0                    7         259  2022      1   
4           0            0                    3         277  2022      1   

   day_of_week  is_month_end  
0            5             0  
1            5             0  
2          

In [ ]:
# Sorting  by date
df = df.sort_values(by='transaction_date')

#index split
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

# Train
X_train = train_df.drop(['items_sold', 'transaction_date'], axis=1)
y_train = train_df['items_sold']
print(X_train.head())
print(y_train.head())

# Test
X_test = test_df.drop(['items_sold', 'transaction_date'], axis=1)
y_test = test_df['items_sold']
print(X_test.head())
print(y_test.head())


#A random split is inappropriate for time-ordered (time series) data because it disrupts the chronological sequence, leading to data leakage and overly optimistic performance metrics. In time-ordered datasets, the target variable depends on previous time steps; therefore, training a model on "future" data to predict "past" data violates the principle of forecasting, where only past data is available to predict future events.

   store_id store_size location_type  promotion_type  is_weekend  is_festival  \
0        28      small    semi-urban       free_gift           1            0   
1         5     medium    semi-urban       free_gift           1            1   
2        13      small    semi-urban  loyalty_points           1            0   
3        17      small         urban       free_gift           1            0   
4        50     medium    semi-urban            bogo           0            0   

   competition_density  year  month  day_of_week  is_month_end  
0                    5  2022      1            5             0  
1                    1  2022      1            5             0  
2                    6  2022      1            6             0  
3                    7  2022      1            6             0  
4                    3  2022      1            0             0  
0    224
1    348
2    249
3    259
4    277
Name: items_sold, dtype: int64
     store_id store_size location_type  promoti

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Categorical columns
cat_cols = ['promotion_type', 'location_type', 'store_size']

# Numerical columns
num_cols = [col for col in X_train.columns if col not in cat_cols]

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), cat_cols)
    ]
)

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import matplotlib.pyplot as plt

# Pipelines
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

# Train
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

# Predict
y_pred_lr = lr_pipeline.predict(X_test)
y_pred_rf = rf_pipeline.predict(X_test)

# Metrics
def evaluate(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    print(f"\n{model_name}")
    print("RMSE:", rmse)
    print("MAE:", mae)

evaluate(y_test, y_pred_lr, "Linear Regression")
evaluate(y_test, y_pred_rf, "Random Forest")


Linear Regression
RMSE: 27.121451164890622
MAE: 21.052926674588395

Random Forest
RMSE: 31.658897860633115
MAE: 24.904708333333335
